# **LIME**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
MODEL_DIR = "/content/drive/MyDrive/models/camelbert_model"
MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-mix"

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

class MultiLabelCAMeLBERT(nn.Module):
    def __init__(self, model_name: str, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden + 1, 3)
        self.emoji_boost = nn.Parameter(torch.tensor(0.75, dtype=torch.float32))

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, emoji_flag=None):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids if token_type_ids is not None else None,
        )

        pooled = out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state[:, 0, :]
        pooled = self.drop(pooled)

        if emoji_flag is None:
            emoji_flag = torch.zeros(pooled.size(0), device=pooled.device, dtype=pooled.dtype)
        else:
            emoji_flag = emoji_flag.to(pooled.dtype)

        x = torch.cat([pooled, emoji_flag.unsqueeze(1)], dim=1)
        logits = self.classifier(x)

        # تأثير الإيموجي على الكراهية فقط
        logits[:, 1] = logits[:, 1] + self.emoji_boost * emoji_flag

        return {"logits": logits}

In [ ]:
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

model = MultiLabelCAMeLBERT(MODEL_NAME).to(device)

model.load_state_dict(
    torch.load(
        os.path.join(MODEL_DIR, "pytorch_model.bin"),
        map_location=device
    )
)

model.eval()

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-mix
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Model loaded successfully ✅


In [ ]:
HATE_EMOJIS = set([
    '💦','🐖','🐷','🐽','👞','🐕','🐶','💩',
    '🐄','🐮','🐑','🐏','👎','😡','🤬','👺','👿','😠'
])

def emoji_flag(text):
    return 1 if any(e in text for e in HATE_EMOJIS) else 0

In [ ]:
import numpy as np

MAX_LENGTH = 128

tasks = {
    0: ["Egyptian", "Saudi"],
    1: ["Not Hate", "Hate"],
    2: ["Political", "Religious"]
}

In [ ]:
def predict_raw(text):
    model.eval()

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(device)

    eflag = torch.tensor([float(emoji_flag(text))], device=device)

    with torch.no_grad():
        out = model(**enc, emoji_flag=eflag)

    probs = torch.sigmoid(out["logits"]).squeeze(0).cpu().numpy()

    return {
        "Dialect": {
            "Prediction": "Saudi" if probs[0] >= 0.5 else "Egyptian",
            "Egyptian": 1 - probs[0],
            "Saudi": probs[0]
        },
        "Hate": {
            "Prediction": "Hate" if probs[1] >= 0.5 else "Not Hate",
            "Not Hate": 1 - probs[1],
            "Hate": probs[1]
        },
        "Topic": {
            "Prediction": "Religious" if probs[2] >= 0.5 else "Political",
            "Political": 1 - probs[2],
            "Religious": probs[2]
        },
        "emoji_flag": int(emoji_flag(text))
    }

In [ ]:
def predict_proba_label(label_index):
    def classifier_fn(texts):
        model.eval()
        outputs = []

        for text in texts:
            enc = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH
            ).to(device)

            eflag = torch.tensor([float(emoji_flag(text))], device=device)

            with torch.no_grad():
                out = model(**enc, emoji_flag=eflag)

            probs = torch.sigmoid(out["logits"]).squeeze(0).cpu().numpy()

            p = probs[label_index]
            outputs.append([1 - p, p])

        return np.array(outputs)

    return classifier_fn

In [ ]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 11.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=00c606f3d44887f3c21211842a6d4be70d6657596631e0fbf2491764489d585f
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


# **Examples from the Test Set**

In [ ]:
from lime.lime_text import LimeTextExplainer
import pandas as pd

# =========================
# Live Explain Function
# =========================
def explain_text_live(text):
    pred = predict_raw(text)

    print("\n" + "="*50)
    print(f"Input text: {text}")
    print("="*50)

    print("\nPrediction:")
    print(f"- Dialect: {pred['Dialect']['Prediction']} "
          f"(Egyptian={pred['Dialect']['Egyptian']:.2f}, Saudi={pred['Dialect']['Saudi']:.2f})")

    print(f"- Hate: {pred['Hate']['Prediction']} "
          f"(Not Hate={pred['Hate']['Not Hate']:.2f}, Hate={pred['Hate']['Hate']:.2f})")

    print(f"- Topic: {pred['Topic']['Prediction']} "
          f"(Political={pred['Topic']['Political']:.2f}, Religious={pred['Topic']['Religious']:.2f})")

    print(f"- Emoji flag: {pred['emoji_flag']}")

    print("\nBrief numeric explanation:")

    # تحديد الفئة المختارة لكل مهمة
    selected_classes = {
        0: 1 if pred["Dialect"]["Prediction"] == "Saudi" else 0,
        1: 1 if pred["Hate"]["Prediction"] == "Hate" else 0,
        2: 1 if pred["Topic"]["Prediction"] == "Religious" else 0
    }

    task_names = {
        0: "Dialect",
        1: "Hate",
        2: "Topic"
    }

    # تفسير كل مهمة
    for label_index, class_pair in tasks.items():
        class_id = selected_classes[label_index]
        class_name = class_pair[class_id]

        explainer = LimeTextExplainer(
            class_names=class_pair,
            split_expression=r"\s+",
            random_state=42   # مهم عشان النتائج ثابتة
        )

        exp = explainer.explain_instance(
            text_instance=text,
            classifier_fn=predict_proba_label(label_index),
            labels=[class_id],
            num_features=5
        )

        df = pd.DataFrame(
            exp.as_list(label=class_id),
            columns=["Feature", "Weight"]
        )

        df["Abs"] = df["Weight"].abs()
        df = df.sort_values("Abs", ascending=False).drop(columns="Abs").head(3)

        print(f"\n{task_names[label_index]} → {class_name}")

        for _, row in df.iterrows():
            direction = "supports" if row["Weight"] > 0 else "opposes"
            print(f"  - {row['Feature']}: {row['Weight']:.4f} ({direction})")


# =========================
# Live Mode
# =========================
print("=== Live Explainable Mode ===")
print("Type a sentence, then press Enter")
print("To exit, type: exit / quit / stop\n")

while True:
    text = input("Tweet> ").strip()

    if text.lower() in ["exit", "quit", "stop"]:
        print("Done.")
        break

    if text == "":
        continue

    explain_text_live(text)

=== Live Explainable Mode ===
Type a sentence, then press Enter
To exit, type: exit / quit / stop

Tweet> انت ماخذ الموضوع ببساطه شديده السياسه يا مشروعي يا انك عدوي وانت تعرف ان السعوديه اليوم مشروعها مختلف المحور الثلاثي لذالك حنا اعداء بالنسبه لهم الزمن تكتشف الشي 

Input text: انت ماخذ الموضوع ببساطه شديده السياسه يا مشروعي يا انك عدوي وانت تعرف ان السعوديه اليوم مشروعها مختلف المحور الثلاثي لذالك حنا اعداء بالنسبه لهم الزمن تكتشف الشي

Prediction:
- Dialect: Saudi (Egyptian=0.00, Saudi=1.00)
- Hate: Hate (Not Hate=0.35, Hate=0.65)
- Topic: Political (Political=0.99, Religious=0.01)
- Emoji flag: 0

Brief numeric explanation:

Dialect → Saudi
  - السعوديه: 0.0763 (supports)
  - حنا: 0.0625 (supports)
  - ماخذ: 0.0533 (supports)

Hate → Hate
  - عدوي: 0.4457 (supports)
  - يا: 0.2720 (supports)
  - السياسه: -0.1567 (opposes)

Topic → Political
  - السعوديه: 0.6078 (supports)
  - حنا: 0.1151 (supports)
  - السياسه: 0.0944 (supports)
Tweet> المره دي الشعب اللي هيفشخهم احنا لسه بندفع ف

# **External Examples for Evaluating Model Performance**

In [ ]:
from lime.lime_text import LimeTextExplainer
import pandas as pd

# =========================
# Live Explain Function
# =========================
def explain_text_live(text):
    pred = predict_raw(text)

    print("\n" + "="*50)
    print(f"Input text: {text}")
    print("="*50)

    print("\nPrediction:")
    print(f"- Dialect: {pred['Dialect']['Prediction']} "
          f"(Egyptian={pred['Dialect']['Egyptian']:.2f}, Saudi={pred['Dialect']['Saudi']:.2f})")

    print(f"- Hate: {pred['Hate']['Prediction']} "
          f"(Not Hate={pred['Hate']['Not Hate']:.2f}, Hate={pred['Hate']['Hate']:.2f})")

    print(f"- Topic: {pred['Topic']['Prediction']} "
          f"(Political={pred['Topic']['Political']:.2f}, Religious={pred['Topic']['Religious']:.2f})")

    print(f"- Emoji flag: {pred['emoji_flag']}")

    print("\nBrief numeric explanation:")

    # تحديد الفئة المختارة لكل مهمة
    selected_classes = {
        0: 1 if pred["Dialect"]["Prediction"] == "Saudi" else 0,
        1: 1 if pred["Hate"]["Prediction"] == "Hate" else 0,
        2: 1 if pred["Topic"]["Prediction"] == "Religious" else 0
    }

    task_names = {
        0: "Dialect",
        1: "Hate",
        2: "Topic"
    }

    # تفسير كل مهمة
    for label_index, class_pair in tasks.items():
        class_id = selected_classes[label_index]
        class_name = class_pair[class_id]

        explainer = LimeTextExplainer(
            class_names=class_pair,
            split_expression=r"\s+",
            random_state=42   # مهم عشان النتائج ثابتة
        )

        exp = explainer.explain_instance(
            text_instance=text,
            classifier_fn=predict_proba_label(label_index),
            labels=[class_id],
            num_features=5
        )

        df = pd.DataFrame(
            exp.as_list(label=class_id),
            columns=["Feature", "Weight"]
        )

        df["Abs"] = df["Weight"].abs()
        df = df.sort_values("Abs", ascending=False).drop(columns="Abs").head(3)

        print(f"\n{task_names[label_index]} → {class_name}")

        for _, row in df.iterrows():
            direction = "supports" if row["Weight"] > 0 else "opposes"
            print(f"  - {row['Feature']}: {row['Weight']:.4f} ({direction})")


# =========================
# Live Mode
# =========================
print("=== Live Explainable Mode ===")
print("Type a sentence, then press Enter")
print("To exit, type: exit / quit / stop\n")

while True:
    text = input("Tweet> ").strip()

    if text.lower() in ["exit", "quit", "stop"]:
        print("Done.")
        break

    if text == "":
        continue

    explain_text_live(text)

=== Live Explainable Mode ===
Type a sentence, then press Enter
To exit, type: exit / quit / stop

Tweet> وربنا اللي بيحصل ده مش طبيعي خالص واللي بيحصل في البلد محتاج وقفة جد 😡

Input text: وربنا اللي بيحصل ده مش طبيعي خالص واللي بيحصل في البلد محتاج وقفة جد 😡

Prediction:
- Dialect: Egyptian (Egyptian=0.98, Saudi=0.02)
- Hate: Not Hate (Not Hate=0.97, Hate=0.03)
- Topic: Political (Political=0.95, Religious=0.05)
- Emoji flag: 1

Brief numeric explanation:

Dialect → Egyptian
  - ده: 0.0091 (supports)
  - خالص: 0.0089 (supports)
  - مش: 0.0083 (supports)

Hate → Not Hate
  - بيحصل: 0.0230 (supports)
  - 😡: -0.0207 (opposes)
  - وربنا: 0.0205 (supports)

Topic → Political
  - البلد: 0.6621 (supports)
  - وربنا: -0.0690 (opposes)
  - بيحصل: 0.0641 (supports)
Tweet> يا أذناب المحور الثلاثي تظنون إنكم بتهزون جبل حنا أدرى بخبثكم يا 🐕 تلبسون ثوب الدين وأنتم 👿 في هيئة بشر السعودية خط أحمر والزمن بيكشف خيانتكم يا عبيد الصهاينة

Input text: يا أذناب المحور الثلاثي تظنون إنكم بتهزون جبل حنا أدر